# 04 — Aggregations and Spark SQL

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Learning goal:** Group data, compute sums and counts, register a temporary view, and compare DataFrame and SQL outputs. The business metric is **revenue from completed sales with valid customer and product references**.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the workshop folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('TG117-Workshop')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

In [ ]:
raw, customers, products = read_raw(spark, ROOT)
clean = clean_transactions(raw)
enriched = completed_enriched(clean, customers, products)
print('Completed, reference-valid sales:', enriched.count())
assert enriched.count() == 36

### DataFrame aggregation

In [ ]:
report = revenue_by_category(enriched)
report.show(truncate=False)
assert str(report.agg(F.sum('revenue')).first()[0]) == '342228.00'

### Spark SQL temporary view

Temporary views belong to the active SparkSession; they are not permanent tables.

In [ ]:
enriched.createOrReplaceTempView('sales_valid')
sql_report = spark.sql("""
SELECT category,
       COUNT(DISTINCT transaction_id) AS orders,
       SUM(line_total) AS revenue
FROM sales_valid
GROUP BY category
ORDER BY category
""")
sql_report.show(truncate=False)
assert {r['category']:str(r['revenue']) for r in sql_report.collect()} == {r['category']:str(r['revenue']) for r in report.collect()}

### Daily summary and group counts

In [ ]:
daily = (enriched.groupBy('order_date')
         .agg(F.count('*').alias('sales_rows'), F.sum('line_total').alias('revenue'))
         .orderBy('order_date'))
daily.show(truncate=False)
enriched.groupBy('city').agg(F.sum('line_total').alias('city_revenue')).orderBy('city').show(truncate=False)

**Check:** Why do cancelled transactions not contribute to completed revenue? How does `COUNT(*)` differ from `COUNT(DISTINCT transaction_id)`?